<a href="https://colab.research.google.com/github/JoseAlberto88/Cleaning-Online-Retail-Data-and-Predicting-Purchase-Quantity/blob/main/Cleaning_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Data Cleaning**

## **Dataset structure and initial profiling**

### •	Import both worksheets and preserve the original data.

In [6]:
!pip install -q --upgrade openpyxl

In [15]:
# We use this command to create a new folder called "data" in which we're going to save the online_retail_II excel document with the two worksheets.
import os

print("Files in data/ folder:")
print(os.listdir('data') if os.path.exists('data') else "data folder does not exist")

import openpyxl
print("openpyxl version:", openpyxl.__version__)

Files in data/ folder:
['online_retail_II.xlsx']
openpyxl version: 3.1.5


In [16]:
# Mount the drive to save the data
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
# Create a data folder in drive
import os
os.makedirs('/content/drive/MyDrive/data', exist_ok=True)

In [19]:
import os

file_path ='/content/drive/MyDrive/data/online_retail_II.xlsx'


print("File size (bytes):", os.path.getsize(file_path))

with open(file_path, 'rb') as f:
    header = f.read(8)
print("Header bytes:", header)

File size (bytes): 45622278
Header bytes: b'PK\x03\x04\x14\x00\x06\x00'


In [20]:
# Let's import all the libraries that we need to work with the data efficiently.

import pandas as pd
import numpy as np

# Creating the path
file_path = '/content/drive/MyDrive/data/online_retail_II.xlsx'

# Later we'll load both sheets (correspondently periods of years 2009-2010 and 2010-2011) into a dict of DataFrames
sheets = pd.read_excel(file_path, sheet_name = None)

# To print the sheet names to see them
print(sheets.keys())

dict_keys(['Year 2009-2010', 'Year 2010-2011'])


### **Why we saved a pickle file**

The raw datsset came as an Excel workbook with two workshhets (one from teh years 2009-2010 and 2010-2011), together holding over a million transcations rows. Loading `.xlsx` files with `openpyxl` is really slow because it parses the file row-by-row rather tahn in optimized buinay form, in our case this took aproximatelly 1:30 minutes just to read the raw data, before any cleaning even began.


We would have to repeat that slow Excel read every single time we came back to the notebook. To avoid this,, we combined both sheets into a single DataFrame and saved it
 as a **pickle file** (`combined_raw.pkl`).

 Pickle fikes store teh DataFrame in Python's native binary format, so:
 - **Loading is nearly instant** compared to re-parsing Excel.
 - **Data types are preserved exactly** (e.g., dates stay as `datetime`, not strings), which avoidshaving to re-infer or -re-convert types every tome.
 - It lets us treat this as a checkpoint: the *raw, unmodified* combined data is saved once, and all futher cleaning stpes can be re-run from this fast-loading starting point without touching the original Excel fule again.

 **Source:** For more information about **pickle file**, visit the following link https://docs.python.org/3/library/pickle.html


### •	Add a Source_Year variable before combining the worksheets.
### •	Combine the two worksheets into one working dataset.


In [21]:
# Creating the two new DataFrames based on the sheets names.

df_year_2009_2010 = sheets['Year 2009-2010']
df_year_2010_2011 = sheets['Year 2010-2011']

# Keep track of the spurce year, creating a new variable called Source_Year
df_year_2009_2010['Source_Year'] = '2009-2010'
df_year_2010_2011['Source_Year'] = '2010-2011'

# Concatenated the two tables (both have the same names columns)

df = pd.concat([df_year_2009_2010, df_year_2010_2011], ignore_index = True)

# Print the shape and the first five obervations
print("The shape of the DataFrame is:", df.shape)
df.head(5)

The shape of the DataFrame is: (1067371, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Source_Year
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-2010


In [22]:
# The next step is to save this DataFrame in a pickle file, so later it will be faster to download it
df.to_pickle('/content/drive/MyDrive/data/combined_raw.pkl')

### •	Report the number of observations and variables.

In [24]:
# To report the number of observtions and varibles, we nend to use the shape attribute

n_observations, n_variables = df.shape
print(f"The total number of observations is: {n_observations}")
print(f"The total number of variables is: {n_variables}")

The total number of observations is: 1067371
The total number of variables is: 9


### •	Examine the data types of all variables.

In [25]:
# To know the data types of all varibales, we use the info() method
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
 8   Source_Year  1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 73.3+ MB
None


### •	Calculate the number and percentage of missing values in each variable.

In [26]:
# To get the percentage od missing values for each variable, we're going to create a DaraFrame in which we generate two columns: "Missing Count" and "Missing %", where one counts the number of missing values and the another gives the percentage, respectivally.

missing_count = df.isna().sum()
missing_percentage = (df.isna().sum() / len(df)) * 100

# Creating the DataFrame
missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_percentage.round(2)
})

# Calling the missing_summary
missing_summary

,Missing Count,Missing %
Invoice,0,0.00
StockCode,0,0.00
Description,4382,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,243007,22.77
Country,0,0.00
Source_Year,0,0.00


### •	Examine the number of unique invoices, products, customers, descriptions, and countries.

In [29]:
# In the same way, we're going to create a new DataFrame called unique_summary in which we will get the unique values for each categorical column using the nunique() method

unique_summary = pd.DataFrame({
    'Unique Count' : [
        df['Invoice'].nunique(),
        df['StockCode'].nunique(),
        df['Customer ID'].nunique(),
        df['Description'].nunique(),
        df['Country'].nunique()
    ]},
    index=[
        'Unique Invoices',
        'Unique Stock Codes',
        'Unique Customers',
        'Unique Descriptions',
        'Unique Countries'
    ]
)

## Calling the new DataFrame
unique_summary

,Unique Count
Unique Invoices,53628
Unique Stock Codes,5305
Unique Customers,5942
Unique Descriptions,5698
Unique Countries,43


### •	Produce descriptive statistics for Quantity and Price.

In [30]:
# For this exercise, we can use simply the describe() method applied to those ctwo  columns
df[['Quantity', 'Price']].describe()

,Quantity,Price
count,1.067371e+06,1.067371e+06
mean,9.938898e+00,4.649388e+00
std,1.727058e+02,1.235531e+02
min,-8.099500e+04,-5.359436e+04
25%,1.000000e+00,1.250000e+00
50%,3.000000e+00,2.100000e+00
75%,1.000000e+01,4.150000e+00
max,8.099500e+04,3.897000e+04


## Dataset Structure and Initial Profiling

### Overview

The combined Online Retail II dataset contains **1,067,371 observations** across **9 variables**, formed by merging two workshefeets (Year 2009-2010 and Year 2010-2011) and adding a `Source_Year` indicator to preserve traceability of each transaction's oorigin.

### Data Types

All variabbvles loaded with types consistent with their expected role (numeric fields for Quantity and Price, datetime for InvoiceDate, and object/string types for identifiers and categorical fields such as Invoice, StockCode, Description, Customer ID, and Country). No immediate type-conversion issues were found at this stage, though Customer ID may warrant conversion to a string or categorical type later, since it functions as an identifier rather than a numeric quantity.

### Missing Values

- **Description**: 4,382 missing values (0.41% of records). This is a small proportion and unlikely to materially affect analysis, but these rows should be reviewed before deciding whetherh to drop or retain them.
- **Customer ID**: 243,007 missing values (22.77% of records). This is a substantial gap, affecting nearly a quarter of all transactions. Missing Customer IDs are a well-documented characteristic of this dataset, often associated with non-account transactions (such as manual entries, adjustments, or sales not tied to a registered customer). This has direct implications for any customer-level analisis and will require an explicit handling decision.
- All other variables (Invoice, StockCodes, Quantity, InvoiceDate, Price, Country, Source_Year) have no missing values.

### Unique Value Counts

- Unique Invoices: 53,628
- Unique Stock Codes: 5,305
- Unique Customers: 5,942
- Unique Descriptions: 5,698
- Unique Countries: 43

Notably, the number of unique Descriptions (5,698) exceeds the number of unique Stock Codes (5,305). Since each prosduct should logically correspond to a single description, this discrepancy suggests inconsistencies in how product descriptions were recorded, such as spelling variants, inconsistent capitalization, or descriptions that changed over time for the same stock code. This is a data-quality issue thast will need to be addressed before using Description as a reliable categorical feature.

### Descriptive Statistics: Quantity and Price

Both Quantity and Price exhibit wide ranges and extreme outliers relative to their central tendency:

- **Quantity**: mean of approximately 9.94, but a minimum of -80,995 and a maximum of 80,995. The standard deviaticon (approximately 172.7) is far larger than the mean, indicating a distribution heavily influenced by extreme values rather than a typical, tisghtly clustered set of transactions.
- **Price**: mean of approximately 4.65, but a minimum of -53,594.36 and a maximum of 38,970. As with Quantity, the standard deviation (approximately 123.6) dwarfsss the mean, pointing to the same pattern of outlier-driven dispersion.

The presence of large negative values in both variables is a strong signal of specific transaction types rather than data entry noise alone:

- Negative Quantity values likely correspond to **returns or cancelled orders** (the dataset is known to mark cancellations with an "C" prefix on the Invoice field).
- Negative Price values are less expected in a standard sales transaction and likely reflect **manual corrections, adjustments, or write-offs** rather than genuine customer purchases.

The gap between the 75th percentile and the maximum value for both variables (for example, Quantity's 75th percentile is 10 versus a maximum of 80,995) further confirms that a small number of extreme records are disproportionately affecting the summary statistics, and that median-based measures are more representative of typical transactions than the mean.

### Summary

This initial profiling confirms several of the data-quality concerns flagged in the assignment background: a meaningful share of missing Customer IDs, minor missingness in Description, inconsistent product descriptions relative to stock codes, and extreme outliers in both Quantity and Price consistent with returns, cancellations, and manual adjustments. These findings directly motivate the cleaning decisions to be addressed in the next stage of the analysis.

## **Missing values**

### •	Identify missing values.

In [31]:
# In the previous section, we have already identifed the number of missing anf percentage. Now, we will just sort those values ascending

missing_summary = missing_summary.sort_values(by='Missing %', ascending=False)
missing_summary

,Missing Count,Missing %
Customer ID,243007,22.77
Description,4382,0.41
Invoice,0,0.00
Quantity,0,0.00
StockCode,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00
Source_Year,0,0.00


### •	Do not create artificial customer identification numbers.

In [ ]:
# No code is necessary here.

### •	Explain whether records with missing Customer ID should be retained or removed for different types of analysis.

The explanation depends heavily on the analysis:

* For **customer-level analysis** (e.g., customer segmentation, repeat pruchase bahavior, customer lifetime value), then these rows ***must be remove*, since you can't attrubute them to any customer.

* For **product-level or transaction-level anbalysus** (e.g., total quantity sold per product, rervenue by country, demand forescasting), then these observations **can be retained**, since te analysis doesn't depend on cusstomer identity.

* Since our final task (see later in tghe next steps) is predicting **Quantity purchased,** you''ll wnat tio think about whwther Custimer ID is a feactureyou plan to use in the model. If so, missing_ID rows may need removal before modeling, otherwisem they can stay.



### •	Determine whether missing product descriptions can be recovered from other records with the same `StockCode`.

In [32]:
# To answer this question, we can do the following steps

# Find the rows (obs) with missing Description
missing_description = df[df['Description'].isna()]
print("The nukber of observations with missing Description:", len(missing_description))

# For those StockCodes, check if other rows (with the same StrockCode) have a non-missing Description
stockcodes_missing_description = missing_description['StockCode'].unique()

recoverable = df[
    (df['StockCode'].isin(stockcodes_missing_description)) &
    (df['Description'].notna())
]

print("Number of StockCodes with missing Descriptio:", len(stockcodes_missing_description))
print("Number of those StockCodes that have a non-missing Description elsewhere:", recoverable['StockCode'].nunique())

The nukber of observations with missing Description: 4382
Number of StockCodes with missing Descriptio: 2451
Number of those StockCodes that have a non-missing Description elsewhere: 2096


In [33]:
# Build a lookup: StockCode -> most common non-missing Description
stockcode_to_desc = (
    df[df['Description'].notna()]
    .groupby('StockCode')['Description']
    .agg(lambda x: x.mode().iloc[0])  # most frequent description for that StockCode
)

# Create a copy of Description to fill, so we preserve the original for comparison/documentation
df['Description_filled'] = df['Description']

# Fill missing descriptions using the lookup, where possible
mask_missing = df['Description'].isna()
df.loc[mask_missing, 'Description_filled'] = df.loc[mask_missing, 'StockCode'].map(stockcode_to_desc)

# Check how many are still missing after recovery
still_missing = df['Description_filled'].isna().sum()
print("Missing Description before recovery:", df['Description'].isna().sum())
print("Missing Description after recovery:", still_missing)

Missing Description before recovery: 4382
Missing Description after recovery: 363
